In [ ]:
import pandas as pd
import yfinance as yf
from fredapi import Fred
from settings import fred_api_key

In [ ]:
# Import df from csv
raw_df = pd.read_csv('raw_data.csv')

In [44]:
## FROM RAW DATA TO df_y

# Load and clean columns
df = raw_df.iloc[1:].copy()
df.columns = df.columns.str.strip()

# Melt to long format: 'Capital IQ Information' holds info, others are tickers
df_long = df.melt(id_vars=['Capital IQ Information'], var_name='Ticker', value_name='Value')

# Clean numeric values
df_long['Value'] = df_long['Value'].str.replace(',', '').str.strip()
df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')

# Filter only median revenue consensus estimates
mask = df_long['Capital IQ Information'].str.contains('Revenue Median Consensus Estimate', na=False)
df_median = df_long[mask].copy()

# Extract forecast year
df_median['Year'] = df_median['Capital IQ Information'].str.extract(r'CY (\d{4})')[0].astype(int)

# Drop unnecessary column and rename for clarity
df_median = df_median.rename(columns={'Value': 'RevenueForecast'})

# Keep only necessary columns and drop missing values
df_y = df_median[['Ticker', 'Year', 'RevenueForecast']].dropna()

print(df_y.head(10))


   Ticker  Year  RevenueForecast
5    AAPL  2025          406.361
6    AAPL  2026          431.829
7    AAPL  2027          459.682
8    AAPL  2028          455.481
9    AAPL  2029          487.125
39    HPE  2025           32.519
40    HPE  2026           34.134
41    HPE  2027           35.133
42    HPE  2028           35.839
43    HPE  2029           35.881


   Ticker    Year  RevenueForecast
5    AAPL  2025.0          406.361
6    AAPL  2026.0          431.829
7    AAPL  2027.0          459.682
8    AAPL  2028.0          455.481
9    AAPL  2029.0          487.125
39    HPE  2025.0           32.519
40    HPE  2026.0           34.134
41    HPE  2027.0           35.133
42    HPE  2028.0           35.839
43    HPE  2029.0           35.881


In [58]:
import pandas as pd
import numpy as np

# Load raw data
raw_df = pd.read_csv('raw_data.csv')

# Clean column names
raw_df.columns = raw_df.columns.str.strip()

# Drop first row if it contains repeated header or irrelevant info (if needed)
# raw_df = raw_df.iloc[1:]  # Uncomment if first row is not data

# Melt into long format:
# id_vars = 'Capital IQ Information'
# value_vars = all tickers
tickers = raw_df.columns.drop('Capital IQ Information')
df_long = raw_df.melt(id_vars='Capital IQ Information', value_vars=tickers,
                      var_name='Ticker', value_name='Value')

# Strip spaces in strings
df_long['Capital IQ Information'] = df_long['Capital IQ Information'].str.strip()
df_long['Value'] = df_long['Value'].astype(str).str.strip()

# Extract Year from 'Capital IQ Information'
df_long['Year'] = df_long['Capital IQ Information'].str.extract(r'CY (\d{4})')
df_long['Year'] = pd.to_numeric(df_long['Year'], errors='coerce')

# Extract FeatureType by removing the year part, e.g. 'Revenue Median Consensus Estimate'
df_long['FeatureType'] = df_long['Capital IQ Information'].str.replace(r'CY \d{4}', '', regex=True).str.strip()


# Function to convert financial string values, including handling negatives in parentheses
def convert_financial_value(val):
    if val in ['n/a', 'NA', '', None, np.nan]:
        return np.nan
    val = val.replace(',', '')
    # Handle negatives in parentheses
    if val.startswith('(') and val.endswith(')'):
        val = '-' + val[1:-1]
    try:
        return float(val)
    except:
        return np.nan


df_long['Value'] = df_long['Value'].apply(convert_financial_value)

# Drop rows with missing Year or Value
df_long = df_long.dropna(subset=['Year', 'Value'])

# Pivot so each feature is a column
df_features = df_long.pivot_table(index=['Ticker', 'Year'],
                                  columns='FeatureType',
                                  values='Value').reset_index()

# Optional: flatten multiindex columns after pivot_table
df_features.columns.name = None

# Rename columns to cleaner names (optional)
df_x = df_features.rename(columns=lambda x: x.replace('Consensus Estimate', '').strip())

print(df_x.head())


  Ticker    Year  EBITDA High  EBITDA Median  Revenue High  Revenue Median
0   AAPL  2025.0      148.459        138.612       431.783         406.361
1   AAPL  2026.0      160.317        145.630       477.463         431.829
2   AAPL  2027.0      172.633        156.754       483.166         459.682
3   AAPL  2028.0      169.973        169.973       479.823         455.481
4   AAPL  2029.0          NaN            NaN       527.912         487.125


In [30]:
# BUILD RISK FREE RATE DF
# --- Set your FRED API key here ---
fred = Fred(api_key=fred_api_key)
gs10 = fred.get_series('GS10', observation_start='2020-01-01', observation_end='2029-12-31')

# Convert to DataFrame
df_gs10 = pd.DataFrame(gs10)
df_gs10.index = pd.to_datetime(df_gs10.index)
df_gs10.columns = ['Risk_Free_Rate']

# Resample to annual average yield
df_risk_free = df_gs10.resample('YE').mean().reset_index()

# Extract Year
df_risk_free['Year'] = df_risk_free['index'].dt.year

# Keep only needed columns and sort
df_risk_free = df_risk_free[['Year', 'Risk_Free_Rate']].sort_values('Year').reset_index(drop=True)

# --- INFLATION DF ----
cpi = fred.get_series('CPIAUCSL', observation_start='2019-12-31', observation_end='2029-12-31')
# Convert to DataFrame
df_cpi = pd.DataFrame(cpi)
df_cpi.index = pd.to_datetime(df_cpi.index)
df_cpi.columns = ['CPI']

# Resample to annual average CPI
df_cpi_annual = df_cpi.resample('YE').mean().reset_index()

# Extract Year
df_cpi_annual['Year'] = df_cpi_annual['index'].dt.year

# Calculate YoY % change in CPI as Inflation Rate
df_cpi_annual['Inflation'] = df_cpi_annual['CPI'].pct_change() * 100

# Keep only Year and Inflation columns, drop first NaN row
df_cpi_final = df_cpi_annual[['Year', 'Inflation']].dropna().reset_index(drop=True)

# ----- GDP DF -----
# Fetch GDP data from FRED
gdp = fred.get_series('GDP', observation_start='2019-12-31', observation_end='2029-12-31')

# Convert to DataFrame
df_gdp = pd.DataFrame(gdp)
df_gdp.index = pd.to_datetime(df_gdp.index)
df_gdp.columns = ['GDP']

# Resample to annual average GDP
df_gdp_annual = df_gdp.resample('YE').mean().reset_index()

# Extract Year
df_gdp_annual['Year'] = df_gdp_annual['index'].dt.year

# Calculate YoY % change in GDP as GDP Growth Rate
df_gdp_annual['GDP_Growth'] = df_gdp_annual['GDP'].pct_change() * 100

# Keep only Year and GDP_Growth columns, drop first NaN row
df_gdp_final = df_gdp_annual[['Year', 'GDP_Growth']].dropna().reset_index(drop=True)


# MERGE DFs

# Merge sequentially on 'Year'
df_macro = df_risk_free.merge(df_cpi_final, on='Year', how='outer') \
                       .merge(df_gdp_final, on='Year', how='outer')

# Sort by Year and reset index
df_macro = df_macro.sort_values('Year').reset_index(drop=True)

# Display results
print(df_macro)

   Year  Risk_Free_Rate  Inflation  GDP_Growth
0  2020        0.894167   0.087287   -2.640343
1  2021        1.442500   4.679118   10.897511
2  2022        2.951667   7.992644    9.820975
3  2023        3.957500   4.127717    6.589857
4  2024        4.208333   2.951606    5.281902
5  2025        4.406667   2.055587    2.662874
